In [72]:
import numpy as np
class LogisticRegression:
    def __init__(self,learning_rate=0.1,iterations=2000):
        self.learning_rate = learning_rate
        self.iterations = iterations
        self.weight = None
        self.bias = None
        self.loss = []

    def sigmoid(self,x):
        return 1/(1+np.exp(-x))

    def fit(self,X,y):
        n_samples,n_features=X.shape
        self.weight = np.zeros(shape=(n_features,1))
        self.bias = 0
        eps = 1e-15
        for i in range(self.iterations):
            #Foward Propagation
            y_pred = np.dot(X,self.weight) + self.bias
            p = self.sigmoid(y_pred)

            #Backward propagation
            cost = -np.mean(y*np.log(p+eps) + (1-y)*np.log(1-p+eps))
            self.loss.append(cost)

            dw = (1/n_samples)*np.dot(X.T,(p - y))
            db = (1/n_samples)*np.sum(p - y)

            self.weight = self.weight - self.learning_rate * dw
            self.bias = self.bias - self.learning_rate * db
            if i % 100 == 0:
                print("Cost =",cost)
            if i ==0:
                print(f"Weight = {self.weight.shape}")
                print(f"Bias = {np.round(self.bias,4)}")

    def predict(self,x):
        y_pred = np.dot(x,self.weight) + self.bias
        return self.sigmoid(y_pred)

    def predict_treshold(self,x,treshold=0.5):
        predictions = self.predict(x)
        return np.where(predictions > treshold, 1, 0)



In [16]:
import pandas as pd
csv_file=pd.read_csv('fertility.csv')
df = pd.DataFrame(csv_file)
df=df.reset_index(drop=True)

encoded_season = pd.get_dummies(df['Season'], prefix='Season',dtype=int)
encoded_season.reset_index(inplace=True)

df['Childish diseases'] = df['Childish diseases'].map({'yes':1,'no':0})
df['Accident or serious trauma'] = df['Accident or serious trauma'].map({'yes':1,'no':0})
df['Surgical intervention'] = df['Surgical intervention'].map({'yes':1,'no':0})
df['High fevers in the last year'] =df['High fevers in the last year'].map({'more than 3 months ago':1,'less than 3 months ago':0})

encoded_frequency_of_alcohol= pd.get_dummies(df['Frequency of alcohol consumption'],prefix='Frequency of alcohol consumption',dtype=int)
encoded_frequency_of_alcohol=encoded_frequency_of_alcohol.reset_index(drop=True)

encoded_smoking_habit = pd.get_dummies(df['Smoking habit'],prefix='Smoking habit',dtype=int)
encoded_smoking_habit=encoded_smoking_habit.reset_index(drop=True)

df['Diagnosis'] = df['Diagnosis'].map({'Normal':0,'Altered':1})
processed_df = pd.concat(
    [
        encoded_season,
        df['Age'],
        df['Childish diseases'],
        df['Accident or serious trauma'],
        df['Surgical intervention'],
        df['High fevers in the last year'],
        encoded_frequency_of_alcohol,
        encoded_smoking_habit,
        df['Number of hours spent sitting per day'],
        df['Diagnosis']
    ],
    axis=1
)
processed_df = processed_df.drop('index', axis=1)
processed_df = processed_df.dropna()
print(processed_df.info(),"\n")
print(processed_df.head(),"\n")


<class 'pandas.core.frame.DataFrame'>
Index: 72 entries, 0 to 99
Data columns (total 19 columns):
 #   Column                                                 Non-Null Count  Dtype  
---  ------                                                 --------------  -----  
 0   Season_fall                                            72 non-null     int64  
 1   Season_spring                                          72 non-null     int64  
 2   Season_summer                                          72 non-null     int64  
 3   Season_winter                                          72 non-null     int64  
 4   Age                                                    72 non-null     int64  
 5   Childish diseases                                      72 non-null     int64  
 6   Accident or serious trauma                             72 non-null     int64  
 7   Surgical intervention                                  72 non-null     int64  
 8   High fevers in the last year                           72

In [59]:
#Divide the data set into training and test sets

def standardize(X):
    mean = np.mean(X, axis=0)
    std = np.std(X, axis=0)
    return (X - mean) / std, mean, std

train_set = processed_df[:60]
test_set = processed_df[60:]


train_set_to_dict=train_set.to_dict()
keys=list(train_set_to_dict.keys())

values =[]
for k in keys:
    values.append(list(train_set_to_dict[k].values()))




X= np.array([
    values[0],
    values[1],
    values[2],
    values[3],
    standardize(values[4])[0],
    values[5],
    values[6],
    values[7],
    values[8],
    values[9],
    values[10],
    values[11],
    values[12],
    values[13],
    values[14],
    values[15],
    values[16],
    standardize(values[17])[0]

]) # samples
y= np.array([
    values[18],
    ]
)
#Test set
test_set_to_dict=test_set.to_dict()
t_keys = list(test_set_to_dict.keys())

t_values =[]
for k in t_keys:
    t_values.append(list(test_set_to_dict[k].values()))

t_X= np.array([
    t_values[0],
    t_values[1],
    t_values[2],
    t_values[3],
    standardize(t_values[4])[0],
    t_values[5],
    t_values[6],
    t_values[7],
    t_values[8],
    t_values[9],
    t_values[10],
    t_values[11],
    t_values[12],
    t_values[13],
    t_values[14],
    t_values[15],
    t_values[16],
    standardize(t_values[17])[0],
])
final_t_X = t_X.T
t_y= np.array([
    t_values[18],
])

print(np.isnan(X).any())
print(np.isinf(X).any())

(1, 60)
(18, 60)
False
False


In [73]:


logisticRegression = LogisticRegression()
logisticRegression.fit(X.T,y.T)

print("\nTest the model with the training dataset\n")
print("Actual: ",t_y)
print("Predicted: ",logisticRegression.predict(final_t_X))

Cost = 0.6931471805599432
Weight = (18, 1)
Bias = -0.035
Cost = 0.37365975067920026
Cost = 0.35549130957020086
Cost = 0.3449920766186048
Cost = 0.33828474793726493
Cost = 0.33370619820608843
Cost = 0.3304282147512859
Cost = 0.32799812075051277
Cost = 0.32614942589374313
Cost = 0.32471532077391646
Cost = 0.3235859628265986
Cost = 0.3226859500169994
Cost = 0.3219617404625197
Cost = 0.3213742612489278
Cost = 0.32089436464575594
Cost = 0.3204999191618641
Cost = 0.32017387896102467
Cost = 0.3199029615482791
Cost = 0.3196767171087271
Cost = 0.31948685815442607

Test the model with the training dataset

Actual:  [[0 0 0 0 0 1 0 0 0 0 0 0]]
Predicted:  [[0.10657286]
 [0.00721027]
 [0.00782366]
 [0.11391923]
 [0.05965332]
 [0.30074825]
 [0.37210685]
 [0.02022066]
 [0.18218224]
 [0.0038507 ]
 [0.05648466]
 [0.05276183]]
